# Stop Signal & Working Memory: Statistical Analysis

Statistical analyses for the stop signal / working memory dual-task study.

## Table of Contents
1. Setup & Data Loading
2. Within-Subject CI Function
3. RM ANOVA: Memory Load Effects on Go Trials
4. RM ANOVA: SSRT by Condition
5. Paired T-Test: Simple Stop vs Dual-Task SSRT
6. BIC Model Comparison
7. Pairwise T-Tests: SSRT Across Conditions
8. RM ANOVA: Probe Accuracy by Memory Load
9. Pairwise T-Tests: Probe Accuracy by Stop Signal Outcome
10. LMM: Prior-Trial Probe Outcome Predicting Stop Success
11. Summary Table for Manuscript

In [1]:
#imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import f_oneway
import scipy.stats as stats

from stop_wm.config import ProjectConfig
from stop_wm.bic_bayes import calculate_bic, calculate_bf10, interpret_bic_delta

# Initialize config with explicit project root
# Go up two levels from notebook location: data/figures/ -> project root
project_root = Path.cwd().parent
config = ProjectConfig(project_root=project_root)


# Pathing
trial_wise_data_wm_path = config.results_dir / 'post_qc_stop_signal_wm_trials.csv'
trial_wise_data_stop_path = config.results_dir / 'post_qc_stop_signal_trials.csv'
subject_wise_metrics_wm_path = config.results_dir / 'post_qc_stop_signal_wm_metrics.csv'
subject_wise_metrics_stop_path = config.results_dir / 'post_qc_stop_signal_metrics.csv'

# Load the data
trial_wise_data_wm = pd.read_csv(trial_wise_data_wm_path)
trial_wise_data_stop = pd.read_csv(trial_wise_data_stop_path)

# Load the metrics (subject-wise)
metrics_data_wm = pd.read_csv(subject_wise_metrics_wm_path)
metrics_data_stop = pd.read_csv(subject_wise_metrics_stop_path)



Key STOPWM_DATADIR not found in /Users/lyndefolsom/research/working_memory_inhibition/experiment_3/.env.


Key STOPWM_RAWDATADIR not found in /Users/lyndefolsom/research/working_memory_inhibition/experiment_3/.env.


## 2. Within-Subject Confidence Interval Function

This function calculates within-subject confidence intervals using the Cousineau (2005) method with Morey (2008) correction. This is appropriate for repeated measures designs where we want to remove between-subject variability.

In [2]:
from stop_wm.within_subject_ci import calculate_within_subject_ci



## 3. Repeated Measures ANOVA: Memory Load Effects on Go Trials

Proper repeated measures ANOVA with:
- Mauchly's test of sphericity
- Greenhouse-Geisser correction (if sphericity violated)
- Partial eta-squared effect size

Test for significant effects of working memory load (0, 2, 4 items) on go trial accuracy and reaction time.

In [3]:
import pingouin as pg

print("="*70)
print("REPEATED MEASURES ANOVA: Memory Load Effects on Go Trials")
print("="*70)

# Prepare data for ANOVA - need participants with complete data for all conditions
complete_participants = metrics_data_wm.dropna(subset=[
    'dual_task_go_wm2_choice_accuracy', 'dual_task_go_wm2_mean_rt',
    'dual_task_go_wm4_choice_accuracy', 'dual_task_go_wm4_mean_rt',
    'dual_task_go_wm6_choice_accuracy', 'dual_task_go_wm6_mean_rt'
])

print(f"\nParticipants with complete data for all conditions: {len(complete_participants)}\n")

# Create long-format dataframe for pingouin
df_long_acc = pd.DataFrame({
    'subject': list(range(len(complete_participants))) * 3,
    'wm_load': ['2'] * len(complete_participants) + ['4'] * len(complete_participants) + ['6'] * len(complete_participants),
    'accuracy': np.concatenate([
        complete_participants['dual_task_go_wm2_choice_accuracy'].values,
        complete_participants['dual_task_go_wm4_choice_accuracy'].values,
        complete_participants['dual_task_go_wm6_choice_accuracy'].values
    ])
})

df_long_rt = pd.DataFrame({
    'subject': list(range(len(complete_participants))) * 3,
    'wm_load': ['2'] * len(complete_participants) + ['4'] * len(complete_participants) + ['6'] * len(complete_participants),
    'rt': np.concatenate([
        complete_participants['dual_task_go_wm2_mean_rt'].values,
        complete_participants['dual_task_go_wm4_mean_rt'].values,
        complete_participants['dual_task_go_wm6_mean_rt'].values
    ])
})

# === ACCURACY RM-ANOVA ===
print("="*70)
print("GO TRIAL ACCURACY")
print("="*70)

# Descriptive statistics
print("\nDescriptive Statistics:")
for load in ['2', '4', '6']:
    data = df_long_acc[df_long_acc['wm_load'] == load]['accuracy']
    print(f"  WM Load {load}: M = {data.mean():.3f}, SD = {data.std():.3f}")

# Run RM-ANOVA with sphericity test
aov_acc = pg.rm_anova(data=df_long_acc, dv='accuracy', within='wm_load', subject='subject', detailed=True, correction=True)

print("\nRepeated Measures ANOVA:")
print(aov_acc.to_string())

# Sphericity test
spher_acc = pg.sphericity(data=df_long_acc, dv='accuracy', within='wm_load', subject='subject')
print(f"\nMauchly's Test of Sphericity:")
print(f"  W = {spher_acc.W:.4f}, p = {spher_acc.pval:.4f}")
if spher_acc.pval < 0.05:
    print("  ⚠️  Sphericity VIOLATED - use Greenhouse-Geisser corrected values")
    print(f"  Greenhouse-Geisser epsilon: {aov_acc['eps'].values[0]:.4f}")
    print(f"  GG-corrected p-value: {aov_acc['p-GG-corr'].values[0]:.6f}")
else:
    print("  ✓ Sphericity assumption met")

# === REACTION TIME RM-ANOVA ===
print("\n" + "="*70)
print("GO TRIAL REACTION TIME")
print("="*70)

# Descriptive statistics
print("\nDescriptive Statistics:")
for load in ['2', '4', '6']:
    data = df_long_rt[df_long_rt['wm_load'] == load]['rt']
    print(f"  WM Load {load}: M = {data.mean():.1f}ms, SD = {data.std():.1f}ms")

# Run RM-ANOVA with sphericity test
aov_rt = pg.rm_anova(data=df_long_rt, dv='rt', within='wm_load', subject='subject', detailed=True, correction=True)

print("\nRepeated Measures ANOVA:")
print(aov_rt.to_string())

# Sphericity test
spher_rt = pg.sphericity(data=df_long_rt, dv='rt', within='wm_load', subject='subject')
print(f"\nMauchly's Test of Sphericity:")
print(f"  W = {spher_rt.W:.4f}, p = {spher_rt.pval:.4f}")
if spher_rt.pval < 0.05:
    print("  ⚠️  Sphericity VIOLATED - use Greenhouse-Geisser corrected values")
    print(f"  Greenhouse-Geisser epsilon: {aov_rt['eps'].values[0]:.4f}")
    print(f"  GG-corrected p-value: {aov_rt['p-GG-corr'].values[0]:.6f}")
else:
    print("  ✓ Sphericity assumption met")

# === BIC MODEL COMPARISON: Go Trial Accuracy (3 WM loads) ===
print("\n" + "=" * 70)
print("BIC MODEL COMPARISON: Go Trial Performance")
print("=" * 70)

acc_matrix = np.column_stack([
    complete_participants['dual_task_go_wm2_choice_accuracy'].values,
    complete_participants['dual_task_go_wm4_choice_accuracy'].values,
    complete_participants['dual_task_go_wm6_choice_accuracy'].values
])
n_obs_acc = acc_matrix.size
resid_null_acc = acc_matrix.flatten() - np.mean(acc_matrix)
resid_full_acc = (acc_matrix - np.mean(acc_matrix, axis=0)).flatten()
bic_null_acc = calculate_bic(resid_null_acc, 1, n_obs_acc)
bic_full_acc = calculate_bic(resid_full_acc, 3, n_obs_acc)
delta_bic_acc = bic_null_acc - bic_full_acc
bf10_acc = calculate_bf10(delta_bic_acc)
print(f"\nGo Accuracy: ΔBIC = {delta_bic_acc:.2f}, BF10 = {bf10_acc:.2f}")
print(f"  {interpret_bic_delta(delta_bic_acc)}")

rt_matrix = np.column_stack([
    complete_participants['dual_task_go_wm2_mean_rt'].values,
    complete_participants['dual_task_go_wm4_mean_rt'].values,
    complete_participants['dual_task_go_wm6_mean_rt'].values
])
n_obs_rt_bic = rt_matrix.size
resid_null_rt_bic = rt_matrix.flatten() - np.mean(rt_matrix)
resid_full_rt_bic = (rt_matrix - np.mean(rt_matrix, axis=0)).flatten()
bic_null_rt_bic = calculate_bic(resid_null_rt_bic, 1, n_obs_rt_bic)
bic_full_rt_bic = calculate_bic(resid_full_rt_bic, 3, n_obs_rt_bic)
delta_bic_rt_go = bic_null_rt_bic - bic_full_rt_bic
bf10_rt_go = calculate_bf10(delta_bic_rt_go)
print(f"\nGo RT: ΔBIC = {delta_bic_rt_go:.2f}, BF10 = {bf10_rt_go:.2f}")
print(f"  {interpret_bic_delta(delta_bic_rt_go)}")



REPEATED MEASURES ANOVA: Memory Load Effects on Go Trials

Participants with complete data for all conditions: 50

GO TRIAL ACCURACY

Descriptive Statistics:
  WM Load 2: M = 0.920, SD = 0.093
  WM Load 4: M = 0.924, SD = 0.083
  WM Load 6: M = 0.923, SD = 0.084

Repeated Measures ANOVA:
    Source        SS  DF        MS         F     p-unc  p-GG-corr       ng2       eps sphericity  W-spher  p-spher
0  wm_load  0.000596   2  0.000298  0.457902  0.633956   0.632629  0.000539  0.993154       True      inf      1.0
1    Error  0.063785  98  0.000651       NaN       NaN        NaN       NaN       NaN        NaN      NaN      NaN

Mauchly's Test of Sphericity:
  W = inf, p = 1.0000
  ✓ Sphericity assumption met

GO TRIAL REACTION TIME

Descriptive Statistics:
  WM Load 2: M = 748.4ms, SD = 131.1ms
  WM Load 4: M = 765.6ms, SD = 129.4ms
  WM Load 6: M = 777.4ms, SD = 127.6ms

Repeated Measures ANOVA:
    Source            SS  DF            MS          F         p-unc  p-GG-corr       ng2   

/Users/lyndefolsom/research/working_memory_inhibition/experiment_3/.venv/lib/python3.12/site-packages/pingouin/distribution.py:1004: RuntimeWarning: divide by zero encountered in scalar divide
  W = np.prod(eig) / (eig.sum() / d) ** d
/Users/lyndefolsom/research/working_memory_inhibition/experiment_3/.venv/lib/python3.12/site-packages/pingouin/distribution.py:1004: RuntimeWarning: divide by zero encountered in scalar divide
  W = np.prod(eig) / (eig.sum() / d) ** d


## 4. Repeated Measures ANOVA: SSRT by Condition

Proper repeated measures ANOVA with:
- Mauchly's test of sphericity
- Greenhouse-Geisser correction (if sphericity violated)
- Post-hoc pairwise comparisons with Bonferroni correction

Run an ANOVA on the SSRT to test for effects of the memory load condition

In [4]:
import pingouin as pg

print("="*70)
print("REPEATED MEASURES ANOVA: SSRT Across All Conditions")
print("="*70)

# Find participants with complete SSRT data for both tasks
participants_both = list(set(metrics_data_stop['prolific_id']) & set(metrics_data_wm['prolific_id']))

# Filter for participants with complete SSRT data
stop_data = metrics_data_stop[metrics_data_stop['prolific_id'].isin(participants_both)]
stop_data = stop_data.dropna(subset=['SSRT'])

wm_data = metrics_data_wm[metrics_data_wm['prolific_id'].isin(participants_both)]
wm_data = wm_data.dropna(subset=['SSRT_wm2', 'SSRT_wm4', 'SSRT_wm6'])

# Get overlapping participants with complete data
common_participants = list(set(stop_data['prolific_id']) & set(wm_data['prolific_id']))
stop_data_complete = stop_data[stop_data['prolific_id'].isin(common_participants)].set_index('prolific_id')
wm_data_complete = wm_data[wm_data['prolific_id'].isin(common_participants)].set_index('prolific_id')

# Align the data by participant
wm_data_complete = wm_data_complete.loc[stop_data_complete.index]

print(f"\nParticipants with complete SSRT data for all conditions: {len(common_participants)}\n")

# Extract SSRT data for each condition
ssrt_simple_stop = stop_data_complete['SSRT'].values
ssrt_wm2 = wm_data_complete['SSRT_wm2'].values
ssrt_wm4 = wm_data_complete['SSRT_wm4'].values
ssrt_wm6 = wm_data_complete['SSRT_wm6'].values

n_subjects = len(common_participants)

# ========= RM-ANOVA: WM LOADS ONLY (2, 4, 6) =========
print("="*70)
print("ANOVA 1: SSRT Across WM Loads (2, 4, 6)")
print("="*70)

# Create long-format dataframe for WM loads only
df_long_wm = pd.DataFrame({
    'subject': list(range(n_subjects)) * 3,
    'wm_load': ['2'] * n_subjects + ['4'] * n_subjects + ['6'] * n_subjects,
    'ssrt': np.concatenate([ssrt_wm2, ssrt_wm4, ssrt_wm6])
})

# Descriptive statistics
print("\nDescriptive Statistics:")
for load in ['2', '4', '6']:
    data = df_long_wm[df_long_wm['wm_load'] == load]['ssrt']
    print(f"  WM Load {load}: M = {data.mean():.1f}ms, SD = {data.std():.1f}ms")

# Run RM-ANOVA
aov_wm = pg.rm_anova(data=df_long_wm, dv='ssrt', within='wm_load', subject='subject', detailed=True, correction=True)

print("\nRepeated Measures ANOVA:")
print(aov_wm.to_string())

# Sphericity test
spher_wm = pg.sphericity(data=df_long_wm, dv='ssrt', within='wm_load', subject='subject')
print(f"\nMauchly's Test of Sphericity:")
print(f"  W = {spher_wm.W:.4f}, p = {spher_wm.pval:.4f}")
if spher_wm.pval < 0.05:
    print("  ⚠️  Sphericity VIOLATED - use Greenhouse-Geisser corrected values")
    print(f"  Greenhouse-Geisser epsilon: {aov_wm['eps'].values[0]:.4f}")
    print(f"  GG-corrected p-value: {aov_wm['p-GG-corr'].values[0]:.6f}")
else:
    print("  ✓ Sphericity assumption met")

# ========= RM-ANOVA: ALL CONDITIONS (Simple Stop + WM Loads) =========
print("\n" + "="*70)
print("ANOVA 2: SSRT Including Simple Stop Signal Task")
print("="*70)

# Create long-format dataframe for all conditions
df_long_all = pd.DataFrame({
    'subject': list(range(n_subjects)) * 4,
    'condition': ['Simple'] * n_subjects + ['WM2'] * n_subjects + ['WM4'] * n_subjects + ['WM6'] * n_subjects,
    'ssrt': np.concatenate([ssrt_simple_stop, ssrt_wm2, ssrt_wm4, ssrt_wm6])
})

# Descriptive statistics
print("\nDescriptive Statistics:")
for cond, label in [('Simple', 'Simple Stop'), ('WM2', 'WM Load 2'), ('WM4', 'WM Load 4'), ('WM6', 'WM Load 6')]:
    data = df_long_all[df_long_all['condition'] == cond]['ssrt']
    print(f"  {label}: M = {data.mean():.1f}ms, SD = {data.std():.1f}ms")

# Run RM-ANOVA
aov_all = pg.rm_anova(data=df_long_all, dv='ssrt', within='condition', subject='subject', detailed=True, correction=True)

print("\nRepeated Measures ANOVA:")
print(aov_all.to_string())

# Sphericity test
spher_all = pg.sphericity(data=df_long_all, dv='ssrt', within='condition', subject='subject')
print(f"\nMauchly's Test of Sphericity:")
print(f"  W = {spher_all.W:.4f}, p = {spher_all.pval:.4f}")
if spher_all.pval < 0.05:
    print("  ⚠️  Sphericity VIOLATED - use Greenhouse-Geisser corrected values")
    print(f"  Greenhouse-Geisser epsilon: {aov_all['eps'].values[0]:.4f}")
    print(f"  GG-corrected p-value: {aov_all['p-GG-corr'].values[0]:.6f}")
else:
    print("  ✓ Sphericity assumption met")

# ========= POST-HOC PAIRWISE COMPARISONS =========
print("\n" + "="*70)
print("Post-Hoc Pairwise Comparisons (Paired T-Tests with Bonferroni)")
print("="*70)

# Use pingouin's pairwise_tests for proper correction
posthoc = pg.pairwise_tests(data=df_long_all, dv='ssrt', within='condition', subject='subject',
                            padjust='bonf', effsize='cohen')
print("\n")
print(posthoc[['Contrast', 'A', 'B', 'T', 'dof', 'p-unc', 'p-corr', 'cohen']].to_string())

# Effect size interpretation
print("\n" + "="*70)
print("Effect Size Interpretation (Cohen's d):")
print("="*70)
print("  |d| < 0.2: negligible")
print("  |d| 0.2-0.5: small")
print("  |d| 0.5-0.8: medium")
print("  |d| > 0.8: large")

REPEATED MEASURES ANOVA: SSRT Across All Conditions

Participants with complete SSRT data for all conditions: 50

ANOVA 1: SSRT Across WM Loads (2, 4, 6)

Descriptive Statistics:
  WM Load 2: M = 255.5ms, SD = 61.9ms
  WM Load 4: M = 258.9ms, SD = 58.7ms
  WM Load 6: M = 266.2ms, SD = 78.5ms

Repeated Measures ANOVA:
    Source             SS  DF           MS         F     p-unc  p-GG-corr      ng2       eps sphericity   W-spher  p-spher
0  wm_load    2985.208257   2  1492.604129  1.168087  0.315252   0.315079  0.00451  0.994263       True  0.994229  0.87032
1    Error  125226.277193  98  1277.819155       NaN       NaN        NaN      NaN       NaN        NaN       NaN      NaN

Mauchly's Test of Sphericity:
  W = 0.9942, p = 0.8703
  ✓ Sphericity assumption met

ANOVA 2: SSRT Including Simple Stop Signal Task

Descriptive Statistics:
  Simple Stop: M = 257.4ms, SD = 105.9ms
  WM Load 2: M = 255.5ms, SD = 61.9ms
  WM Load 4: M = 258.9ms, SD = 58.7ms
  WM Load 6: M = 266.2ms, SD = 78.5

## 5. Paired T-Test: Simple Stop vs Dual-Task SSRT

Paired t-test comparing SSRT between simple stop signal task and dual-task (collapsed across memory loads).

In [5]:
# === PAIRED T-TEST: SIMPLE STOP VS DUAL TASK (COLLAPSED ACROSS WM LOADS) ===
print("\n=== ANOVA: Simple Stop Signal vs Dual Task (All WM Loads Combined) ===")

# Calculate each participant's mean SSRT across all dual task conditions
ssrt_dual_task_mean = wm_data_complete[['SSRT_wm2', 'SSRT_wm4', 'SSRT_wm6']].mean(axis=1).values

# Perform paired t-test (which is equivalent to a 2-level repeated measures ANOVA)
from scipy.stats import ttest_rel
t_stat, p_value = ttest_rel(ssrt_simple_stop, ssrt_dual_task_mean)

# Calculate F-statistic (F = t^2 for 2-level comparison)
f_stat = t_stat ** 2

print(f"Paired t-test (Simple Stop vs Dual Task):")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  F-statistic: {f_stat:.4f} (F = t²)")
print(f"  p-value: {p_value:.6f}")
print(f"  Significant: {'Yes' if p_value < 0.05 else 'No'} (α = 0.05)")

# Calculate Cohen's d for paired samples
diff = ssrt_simple_stop - ssrt_dual_task_mean
cohens_d_value = np.mean(diff) / np.std(diff, ddof=1)
print(f"  Effect size (Cohen's d): {cohens_d_value:.4f} "
      f"({'small' if abs(cohens_d_value) < 0.5 else 'medium' if abs(cohens_d_value) < 0.8 else 'large'} effect)")

# Calculate eta-squared
# For 2-group comparison: η² = t² / (t² + df)
df = len(ssrt_simple_stop) - 1
eta_squared_2group = (t_stat ** 2) / (t_stat ** 2 + df)
print(f"  Effect size (η²): {eta_squared_2group:.4f}")

# Descriptive statistics
print(f"\nDescriptive Statistics:")
print(f"  Simple Stop Signal: M={np.mean(ssrt_simple_stop):.1f}ms, SD={np.std(ssrt_simple_stop, ddof=1):.1f}ms")
print(f"  Dual Task (WM 0+2+4): M={np.mean(ssrt_dual_task_mean):.1f}ms, SD={np.std(ssrt_dual_task_mean, ddof=1):.1f}ms")
print(f"  Mean Difference: {np.mean(ssrt_dual_task_mean) - np.mean(ssrt_simple_stop):+.1f}ms")

print(f"\nInterpretation:")
if p_value < 0.05:
    direction = "slower" if np.mean(ssrt_dual_task_mean) > np.mean(ssrt_simple_stop) else "faster"
    print(f"  ✓ Dual task SSRT is significantly {direction} than Simple Stop Signal")
else:
    print(f"  ✗ No significant difference between Simple Stop and Dual Task SSRT")
    print(f"    The presence of a working memory task does not significantly affect inhibitory control")

# === BIC Model Comparison ===
ssrt_2cond = np.column_stack([ssrt_simple_stop, ssrt_dual_task_mean])
n_obs_2 = ssrt_2cond.size
resid_null_2 = ssrt_2cond.flatten() - np.mean(ssrt_2cond)
resid_full_2 = (ssrt_2cond - np.mean(ssrt_2cond, axis=0)).flatten()
bic_null_2 = calculate_bic(resid_null_2, 1, n_obs_2)
bic_full_2 = calculate_bic(resid_full_2, 2, n_obs_2)
delta_bic_2 = bic_null_2 - bic_full_2
bf10_2 = calculate_bf10(delta_bic_2)
print(f"\nBayesian model comparison: ΔBIC = {delta_bic_2:.2f}, BF10 = {bf10_2:.2f}")
print(f"  {interpret_bic_delta(delta_bic_2)}")




=== ANOVA: Simple Stop Signal vs Dual Task (All WM Loads Combined) ===
Paired t-test (Simple Stop vs Dual Task):
  t-statistic: -0.2178
  F-statistic: 0.0474 (F = t²)
  p-value: 0.828470
  Significant: No (α = 0.05)
  Effect size (Cohen's d): -0.0308 (small effect)
  Effect size (η²): 0.0010

Descriptive Statistics:
  Simple Stop Signal: M=257.4ms, SD=105.9ms
  Dual Task (WM 0+2+4): M=260.2ms, SD=60.3ms
  Mean Difference: +2.8ms

Interpretation:
  ✗ No significant difference between Simple Stop and Dual Task SSRT
    The presence of a working memory task does not significantly affect inhibitory control

Bayesian model comparison: ΔBIC = -4.58, BF10 = 0.10
  Positive evidence against condition effects


## 6. BIC (Bayesian Information Criterion) Model Comparison

Bayesian model comparison for memory load effects on stop signal performance, comparing simple stop signal task vs memory task conditions (0, 2, 4 items). Uses BIC to compare models with and without condition effects.

In [6]:
# === BIC (Bayesian Information Criterion) Model Comparison ===
# Compare models with/without condition effects on Stop Signal Success Rate and GO RT
# Uses calculate_bic, calculate_bf10, interpret_bic_delta from stop_wm.bic_bayes

print("="*70)
print("=== BIC MODEL COMPARISON: Memory Load Effects ===")
print("="*70)

# === PREPARE DATA ===
# Find participants with complete data for both tasks
participants_both = list(set(metrics_data_stop['prolific_id']) & set(metrics_data_wm['prolific_id']))

# Filter for participants with complete stop signal success rate data
stop_data = metrics_data_stop[metrics_data_stop['prolific_id'].isin(participants_both)]
stop_data = stop_data.dropna(subset=['stop_inhibition_success_rate'])

wm_data = metrics_data_wm[metrics_data_wm['prolific_id'].isin(participants_both)]
wm_data = wm_data.dropna(subset=[
    'dual_task_stop_wm2_inhibition_success_rate',
    'dual_task_stop_wm4_inhibition_success_rate',
    'dual_task_stop_wm6_inhibition_success_rate'
])

# Get overlapping participants with complete data
common_participants = list(set(stop_data['prolific_id']) & set(wm_data['prolific_id']))
stop_data_complete = stop_data[stop_data['prolific_id'].isin(common_participants)].set_index('prolific_id')
wm_data_complete = wm_data[wm_data['prolific_id'].isin(common_participants)].set_index('prolific_id')

n_participants = len(common_participants)
print(f"\nParticipants with complete data: {n_participants}")

# ===============================================
# BIC ANALYSIS 1: STOP SIGNAL SUCCESS RATE
# ===============================================
print("\n" + "="*70)
print("BIC ANALYSIS 1: STOP SIGNAL SUCCESS RATE")
print("="*70)

# Create data array: each participant's success rate across 4 conditions
# Conditions: Simple Stop, WM2, WM4, WM6
success_rate_data = np.column_stack([
    stop_data_complete['stop_inhibition_success_rate'].values,
    wm_data_complete['dual_task_stop_wm2_inhibition_success_rate'].values,
    wm_data_complete['dual_task_stop_wm4_inhibition_success_rate'].values,
    wm_data_complete['dual_task_stop_wm6_inhibition_success_rate'].values
])

n_conditions = 4
n_obs = n_participants * n_conditions

# Model 1 (Null): No condition effects - predict grand mean for all
grand_mean_sr = np.mean(success_rate_data)
residuals_null_sr = success_rate_data.flatten() - grand_mean_sr
bic_null_sr = calculate_bic(residuals_null_sr, 1, n_obs)  # 1 parameter (grand mean)

# Model 2 (Full): Condition effects - predict each condition's mean
condition_means_sr = np.mean(success_rate_data, axis=0)
residuals_full_sr = (success_rate_data - condition_means_sr).flatten()
bic_full_sr = calculate_bic(residuals_full_sr, n_conditions, n_obs)  # 4 parameters (4 condition means)

delta_bic_sr = bic_null_sr - bic_full_sr  # Positive = full model better
bf10_sr = calculate_bf10(delta_bic_sr)

print(f"\nCondition means (Stop Signal Success Rate):")
print(f"  Simple Stop:  {condition_means_sr[0]:.4f}")
print(f"  WM Load 2:    {condition_means_sr[1]:.4f}")
print(f"  WM Load 4:    {condition_means_sr[2]:.4f}")
print(f"  WM Load 6:    {condition_means_sr[3]:.4f}")
print(f"  Grand Mean:   {grand_mean_sr:.4f}")

print(f"\nBIC Model Comparison:")
print(f"  Null Model (no condition effects):  BIC = {bic_null_sr:.2f}")
print(f"  Full Model (condition effects):     BIC = {bic_full_sr:.2f}")
print(f"  Delta BIC (Null - Full):            {delta_bic_sr:.2f}")
print(f"  BF10 (Full vs Null):                {bf10_sr:.4f}")
print(f"\n  Interpretation: {interpret_bic_delta(delta_bic_sr)}")

# ===============================================
# BIC ANALYSIS 2: GO TRIAL REACTION TIME
# ===============================================
print("\n" + "="*70)
print("BIC ANALYSIS 2: GO TRIAL REACTION TIME")
print("="*70)

# Get GO RT data
# Filter for participants with complete GO RT data
stop_data_rt = metrics_data_stop[metrics_data_stop['prolific_id'].isin(participants_both)]
stop_data_rt = stop_data_rt.dropna(subset=['go_mean_rt'])

wm_data_rt = metrics_data_wm[metrics_data_wm['prolific_id'].isin(participants_both)]
wm_data_rt = wm_data_rt.dropna(subset=[
    'dual_task_go_wm2_mean_rt',
    'dual_task_go_wm4_mean_rt',
    'dual_task_go_wm6_mean_rt'
])

# Get overlapping participants with complete RT data
common_participants_rt = list(set(stop_data_rt['prolific_id']) & set(wm_data_rt['prolific_id']))
stop_data_rt_complete = stop_data_rt[stop_data_rt['prolific_id'].isin(common_participants_rt)].set_index('prolific_id')
wm_data_rt_complete = wm_data_rt[wm_data_rt['prolific_id'].isin(common_participants_rt)].set_index('prolific_id')

n_participants_rt = len(common_participants_rt)
print(f"\nParticipants with complete GO RT data: {n_participants_rt}")

# Create data array
go_rt_data = np.column_stack([
    stop_data_rt_complete['go_mean_rt'].values,
    wm_data_rt_complete['dual_task_go_wm2_mean_rt'].values,
    wm_data_rt_complete['dual_task_go_wm4_mean_rt'].values,
    wm_data_rt_complete['dual_task_go_wm6_mean_rt'].values
])

n_obs_rt = n_participants_rt * n_conditions

# Model 1 (Null): No condition effects
grand_mean_rt = np.mean(go_rt_data)
residuals_null_rt = go_rt_data.flatten() - grand_mean_rt
bic_null_rt = calculate_bic(residuals_null_rt, 1, n_obs_rt)

# Model 2 (Full): Condition effects
condition_means_rt = np.mean(go_rt_data, axis=0)
residuals_full_rt = (go_rt_data - condition_means_rt).flatten()
bic_full_rt = calculate_bic(residuals_full_rt, n_conditions, n_obs_rt)

delta_bic_rt = bic_null_rt - bic_full_rt
bf10_rt = calculate_bf10(delta_bic_rt)

print(f"\nCondition means (GO RT in ms):")
print(f"  Simple Stop:  {condition_means_rt[0]:.1f}")
print(f"  WM Load 2:    {condition_means_rt[1]:.1f}")
print(f"  WM Load 4:    {condition_means_rt[2]:.1f}")
print(f"  WM Load 6:    {condition_means_rt[3]:.1f}")
print(f"  Grand Mean:   {grand_mean_rt:.1f}")

print(f"\nBIC Model Comparison:")
print(f"  Null Model (no condition effects):  BIC = {bic_null_rt:.2f}")
print(f"  Full Model (condition effects):     BIC = {bic_full_rt:.2f}")
print(f"  Delta BIC (Null - Full):            {delta_bic_rt:.2f}")
print(f"  BF10 (Full vs Null):                {bf10_rt:.4f}")
print(f"\n  Interpretation: {interpret_bic_delta(delta_bic_rt)}")

# ===============================================
# BIC ANALYSIS 3: SSRT (Stop Signal Reaction Time)
# ===============================================
print("\n" + "="*70)
print("BIC ANALYSIS 3: SSRT (STOP SIGNAL REACTION TIME)")
print("="*70)

# Get SSRT data
# Filter for participants with complete SSRT data
stop_data_ssrt = metrics_data_stop[metrics_data_stop['prolific_id'].isin(participants_both)]
stop_data_ssrt = stop_data_ssrt.dropna(subset=['SSRT'])

wm_data_ssrt = metrics_data_wm[metrics_data_wm['prolific_id'].isin(participants_both)]
wm_data_ssrt = wm_data_ssrt.dropna(subset=[
    'SSRT_wm2',
    'SSRT_wm4',
    'SSRT_wm6'
])

# Get overlapping participants with complete SSRT data
common_participants_ssrt = list(set(stop_data_ssrt['prolific_id']) & set(wm_data_ssrt['prolific_id']))
stop_data_ssrt_complete = stop_data_ssrt[stop_data_ssrt['prolific_id'].isin(common_participants_ssrt)].set_index('prolific_id')
wm_data_ssrt_complete = wm_data_ssrt[wm_data_ssrt['prolific_id'].isin(common_participants_ssrt)].set_index('prolific_id')

n_participants_ssrt = len(common_participants_ssrt)
print(f"\nParticipants with complete SSRT data: {n_participants_ssrt}")

# Create data array
ssrt_data = np.column_stack([
    stop_data_ssrt_complete['SSRT'].values,
    wm_data_ssrt_complete['SSRT_wm2'].values,
    wm_data_ssrt_complete['SSRT_wm4'].values,
    wm_data_ssrt_complete['SSRT_wm6'].values
])

n_obs_ssrt = n_participants_ssrt * n_conditions

# Model 1 (Null): No condition effects
grand_mean_ssrt = np.mean(ssrt_data)
residuals_null_ssrt = ssrt_data.flatten() - grand_mean_ssrt
bic_null_ssrt = calculate_bic(residuals_null_ssrt, 1, n_obs_ssrt)

# Model 2 (Full): Condition effects
condition_means_ssrt = np.mean(ssrt_data, axis=0)
residuals_full_ssrt = (ssrt_data - condition_means_ssrt).flatten()
bic_full_ssrt = calculate_bic(residuals_full_ssrt, n_conditions, n_obs_ssrt)

delta_bic_ssrt = bic_null_ssrt - bic_full_ssrt
bf10_ssrt = calculate_bf10(delta_bic_ssrt)

print(f"\nCondition means (SSRT in ms):")
print(f"  Simple Stop:  {condition_means_ssrt[0]:.1f}")
print(f"  WM Load 2:    {condition_means_ssrt[1]:.1f}")
print(f"  WM Load 4:    {condition_means_ssrt[2]:.1f}")
print(f"  WM Load 6:    {condition_means_ssrt[3]:.1f}")
print(f"  Grand Mean:   {grand_mean_ssrt:.1f}")

print(f"\nBIC Model Comparison:")
print(f"  Null Model (no condition effects):  BIC = {bic_null_ssrt:.2f}")
print(f"  Full Model (condition effects):     BIC = {bic_full_ssrt:.2f}")
print(f"  Delta BIC (Null - Full):            {delta_bic_ssrt:.2f}")
print(f"  BF10 (Full vs Null):                {bf10_ssrt:.4f}")
print(f"\n  Interpretation: {interpret_bic_delta(delta_bic_ssrt)}")

# ===============================================
# SUMMARY TABLE
# ===============================================
print("\n" + "="*70)
print("SUMMARY: BIC Model Comparison Results")
print("="*70)
print(f"\n{'Measure':<30} {'BIC Null':>12} {'BIC Full':>12} {'Delta BIC':>12} {'BF10':>12} {'Evidence':<35}")
print("-"*113)
print(f"{'Stop Signal Success Rate':<30} {bic_null_sr:>12.2f} {bic_full_sr:>12.2f} {delta_bic_sr:>12.2f} {bf10_sr:>12.4f} {interpret_bic_delta(delta_bic_sr):<35}")
print(f"{'GO Reaction Time':<30} {bic_null_rt:>12.2f} {bic_full_rt:>12.2f} {delta_bic_rt:>12.2f} {bf10_rt:>12.4f} {interpret_bic_delta(delta_bic_rt):<35}")
print(f"{'SSRT':<30} {bic_null_ssrt:>12.2f} {bic_full_ssrt:>12.2f} {delta_bic_ssrt:>12.2f} {bf10_ssrt:>12.4f} {interpret_bic_delta(delta_bic_ssrt):<35}")

print("\n" + "="*70)
print("Note: Positive Delta BIC indicates evidence FOR condition effects")
print("      (the full model with separate condition means is better)")
print("      BF10 = Bayes Factor favoring Full model over Null")
print("      BF10 > 1: evidence for condition effects")
print("      BF10 < 1: evidence against (1/BF10 = BF in favor of null)")
print("      Interpretation based on Kass & Raftery (1995) guidelines:")
print("      0-2: Weak, 2-6: Positive, 6-10: Strong, >10: Very Strong")
print("      BF approximation from BIC per Wagenmakers (2007)")
print("="*70)

=== BIC MODEL COMPARISON: Memory Load Effects ===

Participants with complete data: 50

BIC ANALYSIS 1: STOP SIGNAL SUCCESS RATE

Condition means (Stop Signal Success Rate):
  Simple Stop:  0.5253
  WM Load 2:    0.5333
  WM Load 4:    0.5346
  WM Load 6:    0.5313
  Grand Mean:   0.5311

BIC Model Comparison:
  Null Model (no condition effects):  BIC = -1386.88
  Full Model (condition effects):     BIC = -1373.66
  Delta BIC (Null - Full):            -13.22
  BF10 (Full vs Null):                0.0013

  Interpretation: Very Strong evidence against condition effects

BIC ANALYSIS 2: GO TRIAL REACTION TIME

Participants with complete GO RT data: 50

Condition means (GO RT in ms):
  Simple Stop:  663.5
  WM Load 2:    748.4
  WM Load 4:    765.6
  WM Load 6:    777.4
  Grand Mean:   738.7

BIC Model Comparison:
  Null Model (no condition effects):  BIC = 1959.67
  Full Model (condition effects):     BIC = 1951.41
  Delta BIC (Null - Full):            8.26
  BF10 (Full vs Null):         

## 7. Pairwise T-Tests: SSRT Across Conditions

Paired t-tests comparing SSRT between each pair of conditions (Simple Stop, WM Load 2, WM Load 4, WM Load 6).

In [7]:
# === PAIRWISE PAIRED T-TESTS: SSRT ACROSS CONDITIONS ===
from itertools import combinations
from scipy.stats import ttest_rel

print("="*80)
print("=== PAIRWISE PAIRED T-TESTS: SSRT ACROSS CONDITIONS ===")
print("="*80)

# === PREPARE DATA ===
# Find participants with complete data for both tasks
participants_both = list(set(metrics_data_stop['prolific_id']) & set(metrics_data_wm['prolific_id']))

# Filter for participants with complete SSRT data
stop_data_ssrt = metrics_data_stop[metrics_data_stop['prolific_id'].isin(participants_both)]
stop_data_ssrt = stop_data_ssrt.dropna(subset=['SSRT'])

wm_data_ssrt = metrics_data_wm[metrics_data_wm['prolific_id'].isin(participants_both)]
wm_data_ssrt = wm_data_ssrt.dropna(subset=['SSRT_wm2', 'SSRT_wm4', 'SSRT_wm6'])

# Get overlapping participants with complete SSRT data
common_participants_ssrt = list(set(stop_data_ssrt['prolific_id']) & set(wm_data_ssrt['prolific_id']))
stop_data_ssrt_complete = stop_data_ssrt[stop_data_ssrt['prolific_id'].isin(common_participants_ssrt)].set_index('prolific_id')
wm_data_ssrt_complete = wm_data_ssrt[wm_data_ssrt['prolific_id'].isin(common_participants_ssrt)].set_index('prolific_id')

# Align indices to ensure paired data
wm_data_ssrt_complete = wm_data_ssrt_complete.loc[stop_data_ssrt_complete.index]

n_participants = len(common_participants_ssrt)
print(f"\nParticipants with complete SSRT data: {n_participants}")

# Create dictionary of SSRT data by condition
ssrt_by_condition = {
    'Simple Stop': stop_data_ssrt_complete['SSRT'].values,
    'WM Load 2': wm_data_ssrt_complete['SSRT_wm2'].values,
    'WM Load 4': wm_data_ssrt_complete['SSRT_wm4'].values,
    'WM Load 6': wm_data_ssrt_complete['SSRT_wm6'].values
}

# Print descriptive statistics
print("\n" + "-"*80)
print("Descriptive Statistics (SSRT in ms):")
print("-"*80)
for condition, data in ssrt_by_condition.items():
    print(f"  {condition:<15}: M = {np.mean(data):7.2f}, SD = {np.std(data, ddof=1):7.2f}")

# === PAIRWISE T-TESTS ===
print("\n" + "="*80)
print("Pairwise Paired T-Tests")
print("="*80)

conditions = list(ssrt_by_condition.keys())
n_comparisons = len(list(combinations(conditions, 2)))
bonferroni_alpha = 0.05 / n_comparisons

print(f"\nNumber of comparisons: {n_comparisons}")
print(f"Bonferroni-corrected alpha: {bonferroni_alpha:.4f}")

print(f"\n{'Comparison':<35} {'t-stat':>10} {'p-value':>12} {'Mean Diff':>12} {'Cohens d':>12} {'Sig':>8}" )
print("-"*91)

results = []
for cond1, cond2 in combinations(conditions, 2):
    data1 = ssrt_by_condition[cond1]
    data2 = ssrt_by_condition[cond2]
    
    # Paired t-test
    t_stat, p_value = ttest_rel(data1, data2)
    
    # Effect size (Cohen's d for paired samples)
    diff = data1 - data2
    mean_diff = np.mean(diff)
    cohens_d = mean_diff / np.std(diff, ddof=1)
    
    # Significance markers
    if p_value < 0.001:
        sig = "***"
    elif p_value < 0.01:
        sig = "**"
    elif p_value < 0.05:
        sig = "*"
    elif p_value < bonferroni_alpha:
        sig = "†"  # Bonferroni significant only
    else:
        sig = ""
    
    comparison = f"{cond1} vs {cond2}"
    print(f"{comparison:<35} {t_stat:>10.3f} {p_value:>12.6f} {mean_diff:>12.2f} {cohens_d:>12.3f} {sig:>8}")
    
    results.append({
        'comparison': comparison,
        't_stat': t_stat,
        'p_value': p_value,
        'mean_diff': mean_diff,
        'cohens_d': cohens_d,
        'sig': sig
    })

print("-"*91)
print("\nSignificance: *** p < .001, ** p < .01, * p < .05")
print(f"              † Bonferroni-corrected (p < {bonferroni_alpha:.4f})")

# === SUMMARY OF SIGNIFICANT EFFECTS ===
print("\n" + "="*80)
print("Summary of Significant Effects (uncorrected p < .05)")
print("="*80)
sig_results = [r for r in results if r['p_value'] < 0.05]
if sig_results:
    for r in sig_results:
        direction = ">" if r['mean_diff'] > 0 else "<"
        conds = r['comparison'].split(' vs ')
        print(f"  {conds[0]} {direction} {conds[1]}: t({n_participants-1}) = {r['t_stat']:.3f}, p = {r['p_value']:.4f}, d = {r['cohens_d']:.3f}")
else:
    print("  No significant pairwise differences found.")

# === BONFERRONI-CORRECTED SUMMARY ===
print("\n" + "="*80)
print(f"Summary of Bonferroni-Corrected Significant Effects (p < {bonferroni_alpha:.4f})")
print("="*80)
bonf_results = [r for r in results if r['p_value'] < bonferroni_alpha]
if bonf_results:
    for r in bonf_results:
        direction = ">" if r['mean_diff'] > 0 else "<"
        conds = r['comparison'].split(' vs ')
        print(f"  {conds[0]} {direction} {conds[1]}: t({n_participants-1}) = {r['t_stat']:.3f}, p = {r['p_value']:.4f}, d = {r['cohens_d']:.3f}")
else:
    print("  No significant pairwise differences survive Bonferroni correction.")

=== PAIRWISE PAIRED T-TESTS: SSRT ACROSS CONDITIONS ===

Participants with complete SSRT data: 50

--------------------------------------------------------------------------------
Descriptive Statistics (SSRT in ms):
--------------------------------------------------------------------------------
  Simple Stop    : M =  257.36, SD =  105.86
  WM Load 2      : M =  255.52, SD =   61.94
  WM Load 4      : M =  258.87, SD =   58.74
  WM Load 6      : M =  266.20, SD =   78.49

Pairwise Paired T-Tests

Number of comparisons: 6
Bonferroni-corrected alpha: 0.0083

Comparison                              t-stat      p-value    Mean Diff     Cohens d      Sig
-------------------------------------------------------------------------------------------
Simple Stop vs WM Load 2                 0.152     0.879870         1.85        0.021         
Simple Stop vs WM Load 4                -0.103     0.918559        -1.50       -0.015         
Simple Stop vs WM Load 6                -0.630     0.53150

## 8. Repeated Measures ANOVA: Probe Accuracy by Memory Load

Proper repeated measures ANOVA with:
- Mauchly's test of sphericity
- Greenhouse-Geisser correction (if sphericity violated)
- Post-hoc pairwise comparisons with Bonferroni correction

One-way repeated measures ANOVA testing the effect of working memory load (0, 2, 4 items) on probe (memory recognition) accuracy.

In [8]:
# === REPEATED MEASURES ANOVA: PROBE ACCURACY BY MEMORY LOAD ===
import pingouin as pg

print("="*70)
print("REPEATED MEASURES ANOVA: Probe Accuracy by Memory Load")
print("="*70)

# Prepare data for ANOVA - need participants with complete data for all conditions
complete_participants_probe = metrics_data_wm.dropna(subset=[
    'probe_wm2_response_accuracy',
    'probe_wm4_response_accuracy',
    'probe_wm6_response_accuracy'
])

n_participants = len(complete_participants_probe)
print(f"\nParticipants with complete probe accuracy data for all conditions: {n_participants}")

# Create long-format dataframe for pingouin
df_long_probe = pd.DataFrame({
    'subject': list(range(n_participants)) * 3,
    'wm_load': ['2'] * n_participants + ['4'] * n_participants + ['6'] * n_participants,
    'accuracy': np.concatenate([
        complete_participants_probe['probe_wm2_response_accuracy'].values,
        complete_participants_probe['probe_wm4_response_accuracy'].values,
        complete_participants_probe['probe_wm6_response_accuracy'].values
    ])
})

# === DESCRIPTIVE STATISTICS ===
print("\n" + "-"*70)
print("Descriptive Statistics:")
print("-"*70)
for load in ['2', '4', '6']:
    data = df_long_probe[df_long_probe['wm_load'] == load]['accuracy']
    print(f"  WM Load {load}: M = {data.mean():.4f}, SD = {data.std():.4f}")

# === REPEATED MEASURES ANOVA ===
print("\n" + "="*70)
print("Repeated Measures ANOVA Results:")
print("="*70)

aov_probe = pg.rm_anova(data=df_long_probe, dv='accuracy', within='wm_load', subject='subject', detailed=True, correction=True)
print("\n")
print(aov_probe.to_string())

# Sphericity test
spher_probe = pg.sphericity(data=df_long_probe, dv='accuracy', within='wm_load', subject='subject')
print(f"\nMauchly's Test of Sphericity:")
print(f"  W = {spher_probe.W:.4f}, p = {spher_probe.pval:.4f}")

# Get the relevant p-value based on sphericity
if spher_probe.pval < 0.05:
    print("  ⚠️  Sphericity VIOLATED - use Greenhouse-Geisser corrected values")
    print(f"  Greenhouse-Geisser epsilon: {aov_probe['eps'].values[0]:.4f}")
    gg_p = aov_probe['p-GG-corr'].values[0]
    print(f"  GG-corrected p-value: {gg_p:.6f}")
    p_value = gg_p
else:
    print("  ✓ Sphericity assumption met")
    p_value = aov_probe['p-unc'].values[0]

# Effect size interpretation
ng2 = aov_probe['ng2'].values[0]
# Calculate partial eta-squared from SS: np2 = SS_effect / (SS_effect + SS_error)
ss_effect = aov_probe['SS'].values[0]
ss_error = aov_probe['SS'].values[1]
np2 = ss_effect / (ss_effect + ss_error) if (ss_effect + ss_error) > 0 else 0

if np2 < 0.01:
    effect_interp = "negligible"
elif np2 < 0.06:
    effect_interp = "small"
elif np2 < 0.14:
    effect_interp = "medium"
else:
    effect_interp = "large"

print(f"\nEffect Size:")
print(f"  Partial η² = {np2:.4f} ({effect_interp})")
print(f"  Generalized η² = {ng2:.4f}")

# === POST-HOC PAIRWISE COMPARISONS ===
print("\n" + "="*70)
print("Post-Hoc Pairwise Comparisons (Paired T-Tests with Bonferroni):")
print("="*70)

posthoc = pg.pairwise_tests(data=df_long_probe, dv='accuracy', within='wm_load', subject='subject',
                            padjust='bonf', effsize='cohen')
print("\n")
print(posthoc[['Contrast', 'A', 'B', 'T', 'dof', 'p-unc', 'p-corr', 'cohen']].to_string())

# === SUMMARY ===
print("\n" + "="*70)
print("Summary:")
print("="*70)
f_stat = aov_probe['F'].values[0]
df1 = int(aov_probe['DF'].values[0])
df2 = int(aov_probe['DF'].values[1])

if p_value < 0.05:
    print(f"  There IS a significant effect of memory load on probe accuracy,")
    print(f"  F({df1}, {df2}) = {f_stat:.3f}, p = {p_value:.4f}, partial η² = {np2:.3f} ({effect_interp})")
else:
    print(f"  There is NO significant effect of memory load on probe accuracy,")
    print(f"  F({df1}, {df2}) = {f_stat:.3f}, p = {p_value:.4f}, partial η² = {np2:.3f} ({effect_interp})")

# === BIC Model Comparison: Probe Accuracy (3 WM loads) ===
print("\n" + "=" * 70)
print("BIC MODEL COMPARISON: Probe Accuracy")
print("=" * 70)

probe_matrix = np.column_stack([
    complete_participants_probe['probe_wm2_response_accuracy'].values,
    complete_participants_probe['probe_wm4_response_accuracy'].values,
    complete_participants_probe['probe_wm6_response_accuracy'].values
])
n_obs_probe_bic = probe_matrix.size
resid_null_probe = probe_matrix.flatten() - np.mean(probe_matrix)
resid_full_probe = (probe_matrix - np.mean(probe_matrix, axis=0)).flatten()
bic_null_probe = calculate_bic(resid_null_probe, 1, n_obs_probe_bic)
bic_full_probe = calculate_bic(resid_full_probe, 3, n_obs_probe_bic)
delta_bic_probe = bic_null_probe - bic_full_probe
bf10_probe = calculate_bf10(delta_bic_probe)
print(f"\nProbe Accuracy: ΔBIC = {delta_bic_probe:.2f}, BF10 = {bf10_probe:.2f}")
print(f"  {interpret_bic_delta(delta_bic_probe)}")

REPEATED MEASURES ANOVA: Probe Accuracy by Memory Load

Participants with complete probe accuracy data for all conditions: 50

----------------------------------------------------------------------
Descriptive Statistics:
----------------------------------------------------------------------
  WM Load 2: M = 0.9071, SD = 0.0881
  WM Load 4: M = 0.8932, SD = 0.0933
  WM Load 6: M = 0.8100, SD = 0.1060

Repeated Measures ANOVA Results:




    Source        SS  DF        MS          F         p-unc     p-GG-corr       ng2       eps sphericity      W-spher  p-spher
0  wm_load  0.275657   2  0.137828  79.239639  3.361367e-21  1.720645e-17  0.168758  0.802155       True  1536.560566      1.0
1    Error  0.170460  98  0.001739        NaN           NaN           NaN       NaN       NaN        NaN          NaN      NaN

Mauchly's Test of Sphericity:
  W = 1536.5606, p = 1.0000
  ✓ Sphericity assumption met

Effect Size:
  Partial η² = 0.6179 (large)
  Generalized η² = 0.1688

Post-Hoc Pairwise Comparisons (Paired T-Tests with Bonferroni):


  Contrast  A  B          T   dof         p-unc        p-corr     cohen
0  wm_load  2  4   1.970463  49.0  5.444913e-02  1.633474e-01  0.153028
1  wm_load  2  6   9.520302  49.0  1.003086e-12  3.009258e-12  0.996103
2  wm_load  4  6  11.212312  49.0  3.939977e-15  1.181993e-14  0.833052

Summary:
  There IS a significant effect of memory load on probe accuracy,
  F(2, 98) = 79.240, p = 0.

## 9. Pairwise T-Tests: Probe Accuracy by Stop Signal Outcome

Paired t-tests comparing probe (memory recognition) accuracy conditional on stop signal outcome:
- **Successful Stop**: Trials where a stop signal was presented and the participant successfully inhibited their response
- **Failed Stop**: Trials where a stop signal was presented but the participant failed to inhibit
- **Go Trials**: Trials with no stop signal

This examines whether successful response inhibition is associated with better or worse memory performance.

Comparisons are performed:
1. Overall (collapsed across WM load)
2. Within each WM load level (0, 2, 4 items)

In [9]:
# === PAIRWISE PAIRED T-TESTS: PROBE ACCURACY BY STOP SIGNAL OUTCOME ===
from scipy.stats import ttest_rel

print("=" * 70)
print("PAIRWISE T-TESTS: Probe Accuracy by Stop Signal Outcome")
print("=" * 70)

# --- Overall Analysis (collapsed across WM load) ---
print("\n--- OVERALL (collapsed across WM load) ---\n")

overall_successful = metrics_data_wm['probe_accuracy_on_successful_stop'].dropna()
overall_failed = metrics_data_wm['probe_accuracy_on_failed_stop'].dropna()
overall_go = metrics_data_wm['probe_accuracy_on_go_trials'].dropna()

print(f"Successful Stop: M = {overall_successful.mean():.3f}, SD = {overall_successful.std():.3f}, n = {len(overall_successful)}")
print(f"Failed Stop:     M = {overall_failed.mean():.3f}, SD = {overall_failed.std():.3f}, n = {len(overall_failed)}")
print(f"Go Trials:       M = {overall_go.mean():.3f}, SD = {overall_go.std():.3f}, n = {len(overall_go)}")

print("\nPaired t-tests (overall):")

def _bic_paired(g1, g2):
    data = np.column_stack([g1, g2])
    n_obs = data.size
    resid_null = data.flatten() - np.mean(data)
    resid_full = (data - np.mean(data, axis=0)).flatten()
    bic_n = calculate_bic(resid_null, 1, n_obs)
    bic_f = calculate_bic(resid_full, 2, n_obs)
    d_bic = bic_n - bic_f
    return d_bic, calculate_bf10(d_bic)

# Successful vs Failed
common_idx = overall_successful.index.intersection(overall_failed.index)
if len(common_idx) > 2:
    t, p = ttest_rel(overall_successful.loc[common_idx], overall_failed.loc[common_idx])
    diff = overall_successful.loc[common_idx] - overall_failed.loc[common_idx]
    d = diff.mean() / diff.std() if diff.std() > 0 else np.nan
    dbic, bf = _bic_paired(overall_successful.loc[common_idx].values, overall_failed.loc[common_idx].values)
    print(f"  Successful vs Failed: t({len(common_idx)-1}) = {t:.3f}, p = {p:.4f}, Cohen's d = {d:.3f}, ΔBIC = {dbic:.2f}, BF10 = {bf:.2f}")

# Successful vs Go
common_idx = overall_successful.index.intersection(overall_go.index)
if len(common_idx) > 2:
    t, p = ttest_rel(overall_successful.loc[common_idx], overall_go.loc[common_idx])
    diff = overall_successful.loc[common_idx] - overall_go.loc[common_idx]
    d = diff.mean() / diff.std() if diff.std() > 0 else np.nan
    dbic, bf = _bic_paired(overall_successful.loc[common_idx].values, overall_go.loc[common_idx].values)
    print(f"  Successful vs Go:     t({len(common_idx)-1}) = {t:.3f}, p = {p:.4f}, Cohen's d = {d:.3f}, ΔBIC = {dbic:.2f}, BF10 = {bf:.2f}")

# Failed vs Go
common_idx = overall_failed.index.intersection(overall_go.index)
if len(common_idx) > 2:
    t, p = ttest_rel(overall_failed.loc[common_idx], overall_go.loc[common_idx])
    diff = overall_failed.loc[common_idx] - overall_go.loc[common_idx]
    d = diff.mean() / diff.std() if diff.std() > 0 else np.nan
    dbic, bf = _bic_paired(overall_failed.loc[common_idx].values, overall_go.loc[common_idx].values)
    print(f"  Failed vs Go:         t({len(common_idx)-1}) = {t:.3f}, p = {p:.4f}, Cohen's d = {d:.3f}, ΔBIC = {dbic:.2f}, BF10 = {bf:.2f}")

# --- By WM Load Level ---
print("\n" + "=" * 70)
print("BY MEMORY LOAD LEVEL")
print("=" * 70)

wm_loads = [2, 4, 6]

for wm_load in wm_loads:
    print(f"\n--- WM Load = {wm_load} ---\n")

    successful = metrics_data_wm[f'probe_wm{wm_load}_accuracy_on_successful_stop'].dropna()
    failed = metrics_data_wm[f'probe_wm{wm_load}_accuracy_on_failed_stop'].dropna()
    go = metrics_data_wm[f'probe_wm{wm_load}_accuracy_on_go_trials'].dropna()

    n_successful = metrics_data_wm[f'probe_wm{wm_load}_n_successful_stop_trials'].dropna()
    n_failed = metrics_data_wm[f'probe_wm{wm_load}_n_failed_stop_trials'].dropna()
    n_go = metrics_data_wm[f'probe_wm{wm_load}_n_go_trials'].dropna()

    print(f"Successful Stop: M = {successful.mean():.3f}, SD = {successful.std():.3f}, n_subj = {len(successful)}, avg_trials = {n_successful.mean():.1f}")
    print(f"Failed Stop:     M = {failed.mean():.3f}, SD = {failed.std():.3f}, n_subj = {len(failed)}, avg_trials = {n_failed.mean():.1f}")
    print(f"Go Trials:       M = {go.mean():.3f}, SD = {go.std():.3f}, n_subj = {len(go)}, avg_trials = {n_go.mean():.1f}")

    # Successful vs Failed
    common_idx = successful.index.intersection(failed.index)
    if len(common_idx) > 2:
        t, p = ttest_rel(successful.loc[common_idx], failed.loc[common_idx])
        diff = successful.loc[common_idx] - failed.loc[common_idx]
        d = diff.mean() / diff.std() if diff.std() > 0 else np.nan
        dbic, bf = _bic_paired(successful.loc[common_idx].values, failed.loc[common_idx].values)
        print(f"\n  Successful vs Failed: t({len(common_idx)-1}) = {t:.3f}, p = {p:.4f}, Cohen's d = {d:.3f}, ΔBIC = {dbic:.2f}, BF10 = {bf:.2f}")

    # Successful vs Go
    common_idx = successful.index.intersection(go.index)
    if len(common_idx) > 2:
        t, p = ttest_rel(successful.loc[common_idx], go.loc[common_idx])
        diff = successful.loc[common_idx] - go.loc[common_idx]
        d = diff.mean() / diff.std() if diff.std() > 0 else np.nan
        dbic, bf = _bic_paired(successful.loc[common_idx].values, go.loc[common_idx].values)
        print(f"  Successful vs Go:     t({len(common_idx)-1}) = {t:.3f}, p = {p:.4f}, Cohen's d = {d:.3f}, ΔBIC = {dbic:.2f}, BF10 = {bf:.2f}")

    # Failed vs Go
    common_idx = failed.index.intersection(go.index)
    if len(common_idx) > 2:
        t, p = ttest_rel(failed.loc[common_idx], go.loc[common_idx])
        diff = failed.loc[common_idx] - go.loc[common_idx]
        d = diff.mean() / diff.std() if diff.std() > 0 else np.nan
        dbic, bf = _bic_paired(failed.loc[common_idx].values, go.loc[common_idx].values)
        print(f"  Failed vs Go:         t({len(common_idx)-1}) = {t:.3f}, p = {p:.4f}, Cohen's d = {d:.3f}, ΔBIC = {dbic:.2f}, BF10 = {bf:.2f}")

PAIRWISE T-TESTS: Probe Accuracy by Stop Signal Outcome

--- OVERALL (collapsed across WM load) ---

Successful Stop: M = 0.875, SD = 0.100, n = 50
Failed Stop:     M = 0.863, SD = 0.083, n = 50
Go Trials:       M = 0.871, SD = 0.093, n = 50

Paired t-tests (overall):
  Successful vs Failed: t(49) = 1.569, p = 0.1231, Cohen's d = 0.222, ΔBIC = -4.16, BF10 = 0.12
  Successful vs Go:     t(49) = 0.589, p = 0.5586, Cohen's d = 0.083, ΔBIC = -4.56, BF10 = 0.10
  Failed vs Go:         t(49) = -1.321, p = 0.1927, Cohen's d = -0.187, ΔBIC = -4.39, BF10 = 0.11

BY MEMORY LOAD LEVEL

--- WM Load = 2 ---

Successful Stop: M = 0.906, SD = 0.115, n_subj = 50, avg_trials = 25.6
Failed Stop:     M = 0.902, SD = 0.092, n_subj = 50, avg_trials = 22.4
Go Trials:       M = 0.909, SD = 0.091, n_subj = 50, avg_trials = 96.0

  Successful vs Failed: t(49) = 0.271, p = 0.7873, Cohen's d = 0.038, ΔBIC = -4.57, BF10 = 0.10
  Successful vs Go:     t(49) = -0.265, p = 0.7923, Cohen's d = -0.037, ΔBIC = -4.59, B

## 10. Linear Mixed Effects Model: Prior-Trial Probe Outcome Predicting Stop Success

Corresponds to Section 16 in the visualization notebook. Trial-level linear mixed model testing whether prior-trial probe outcome (t-1 correct vs. incorrect) predicts stop success on the current trial, with **subject as random intercept**. Overall statistic only (no breakdown by WM load).

- **DV (binary):** stop success (1 = successful inhibition, 0 = failed)
- **IV (binary):** previous-trial probe correct (1 = correct, 0 = incorrect)
- **Random effect:** participant (random intercept only)

Only consecutive trials *within the same block* are included (no cross-block lag).

In [10]:
# === LINEAR MIXED EFFECTS MODEL: Prior-Trial Probe Outcome Predicting Stop Success ===
# Corresponds to Section 16 in visualization notebook.
# Trial-level LMM: prior trial probe outcome (correct vs incorrect) predicts stop success,
# with subject as random intercept. Overall statistic only (no breakdown by WM load).

from statsmodels.regression.mixed_linear_model import MixedLM

trial_data = trial_wise_data_wm.copy()
trial_data = trial_data.sort_values(['participant_id', 'block_num', 'current_trial']).reset_index(drop=True)

# Lag the probe accuracy within each subject × block
trial_data['prev_probe_correct'] = (
    trial_data
    .groupby(['participant_id', 'block_num'])['memory_recognition_correct_trial']
    .shift(1)
)

# Determine stop success: on stop trials, correct_trial == 1 means successful inhibition
trial_data['is_stop'] = (trial_data['stop_trial_SS_trial_type'] == 'stop')
trial_data['stop_success'] = (
    trial_data['is_stop'] & (trial_data['stop_trial_correct_trial'] == 1)
).astype(float)

# Keep only stop trials that have a valid previous-trial probe outcome
stop_with_lag = trial_data.loc[
    trial_data['is_stop'] & trial_data['prev_probe_correct'].notna()
].copy()

# Create numeric subject index for MixedLM (requires integer groups)
stop_with_lag = stop_with_lag.copy()
stop_with_lag['subject_idx'] = pd.Categorical(stop_with_lag['participant_id']).codes

print("=" * 70)
print("LINEAR MIXED EFFECTS MODEL: Prior-Trial Probe Outcome → Stop Success")
print("=" * 70)
print(f"\nTrial-level analysis (overall, collapsed across WM load)")
print(f"N trials: {len(stop_with_lag)}, N subjects: {stop_with_lag['participant_id'].nunique()}")
print(f"  After t-1 correct:   {stop_with_lag['stop_success'][stop_with_lag['prev_probe_correct']==1].mean():.3f} stop success rate")
print(f"  After t-1 incorrect: {stop_with_lag['stop_success'][stop_with_lag['prev_probe_correct']==0].mean():.3f} stop success rate")

# Linear mixed model: stop_success ~ prev_probe_correct + (1|subject)
model = MixedLM.from_formula(
    "stop_success ~ prev_probe_correct",
    data=stop_with_lag,
    groups=stop_with_lag["subject_idx"]
)
result = model.fit(method='lbfgs')

print("\n" + "=" * 70)
print("MODEL RESULTS")
print("=" * 70)
print(result.summary())

# Extract key statistics for reporting
coef = result.params['prev_probe_correct']
se = result.bse['prev_probe_correct']
z = result.tvalues['prev_probe_correct']
p = result.pvalues['prev_probe_correct']
print(f"\n--- Key result ---")
print(f"Prior-trial probe correct (1=correct, 0=incorrect) predicting stop success:")
print(f"  β = {coef:.4f}, SE = {se:.4f}, z = {z:.3f}, p = {p:.4f}")
print(f"  Interpretation: When prior trial probe was INCORRECT (vs correct), stop success rate")
print(f"  is {abs(coef):.3f} {'higher' if coef < 0 else 'lower'} (linear probability model).")

LINEAR MIXED EFFECTS MODEL: Prior-Trial Probe Outcome → Stop Success

Trial-level analysis (overall, collapsed across WM load)
N trials: 7009, N subjects: 50
  After t-1 correct:   0.534 stop success rate
  After t-1 incorrect: 0.532 stop success rate



MODEL RESULTS
           Mixed Linear Model Regression Results
Model:              MixedLM Dependent Variable: stop_success
No. Observations:   7009    Method:             REML        
No. Groups:         50      Scale:              0.2489      
Min. group size:    135     Log-Likelihood:     -5078.5470  
Max. group size:    143     Converged:          Yes         
Mean group size:    140.2                                   
------------------------------------------------------------
                   Coef. Std.Err.   z    P>|z| [0.025 0.975]
------------------------------------------------------------
Intercept          0.532    0.016 33.043 0.000  0.501  0.564
prev_probe_correct 0.002    0.017  0.092 0.927 -0.032  0.036
Group Var          0.000    0.001                           


--- Key result ---
Prior-trial probe correct (1=correct, 0=incorrect) predicting stop success:
  β = 0.0016, SE = 0.0173, z = 0.092, p = 0.9268
  Interpretation: When prior trial probe was INCORRECT (vs

/Users/lyndefolsom/research/working_memory_inhibition/experiment_3/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
/Users/lyndefolsom/research/working_memory_inhibition/experiment_3/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## 11. Summary Table for Manuscript

A comprehensive summary of all statistical analyses for the results write-up.

In [11]:
# === SUMMARY TABLE FOR MANUSCRIPT ===
import pingouin as pg
from scipy.stats import ttest_rel

# Reload data to ensure we have latest
metrics_data_wm = pd.read_csv('../data/results/post_qc_stop_signal_wm_metrics.csv')
metrics_data_stop = pd.read_csv('../data/results/post_qc_stop_signal_metrics.csv')

print("=" * 90)
print("STATISTICAL SUMMARY TABLE FOR MANUSCRIPT")
print("=" * 90)

results = []

# ============================================================================
# 1. GO TRIAL ACCURACY - RM ANOVA
# ============================================================================
complete_go = metrics_data_wm.dropna(subset=[
    'dual_task_go_wm2_choice_accuracy', 'dual_task_go_wm4_choice_accuracy', 'dual_task_go_wm6_choice_accuracy'
])
n = len(complete_go)

df_acc = pd.DataFrame({
    'subject': list(range(n)) * 3,
    'wm_load': ['2'] * n + ['4'] * n + ['6'] * n,
    'accuracy': np.concatenate([
        complete_go['dual_task_go_wm2_choice_accuracy'].values,
        complete_go['dual_task_go_wm4_choice_accuracy'].values,
        complete_go['dual_task_go_wm6_choice_accuracy'].values
    ])
})

aov_acc = pg.rm_anova(data=df_acc, dv='accuracy', within='wm_load', subject='subject', detailed=True, correction=True)
spher_acc = pg.sphericity(data=df_acc, dv='accuracy', within='wm_load', subject='subject')
p_acc = aov_acc['p-GG-corr'].values[0] if spher_acc.pval < 0.05 else aov_acc['p-unc'].values[0]
np2_acc = aov_acc['SS'].values[0] / (aov_acc['SS'].values[0] + aov_acc['SS'].values[1])

mse_acc = aov_acc['MS'].values[1]  # MS for Error row
results.append({
    'Analysis': 'Go Trial Accuracy × WM Load',
    'Test': 'RM-ANOVA',
    'F': f"{aov_acc['F'].values[0]:.2f}",
    'df': f"({int(aov_acc['DF'].values[0])}, {int(aov_acc['DF'].values[1])})",
    'MSE': f"{mse_acc:.4f}",
    'p': f"{p_acc:.4f}",
    'Effect Size': f"η²p = {np2_acc:.3f}",
    'Sphericity (W, p)': f"W={spher_acc.W:.3f}, p={spher_acc.pval:.3f}",
    'n': n
})

# ============================================================================
# 2. GO TRIAL RT - RM ANOVA
# ============================================================================
complete_rt = metrics_data_wm.dropna(subset=[
    'dual_task_go_wm2_mean_rt', 'dual_task_go_wm4_mean_rt', 'dual_task_go_wm6_mean_rt'
])
n = len(complete_rt)

df_rt = pd.DataFrame({
    'subject': list(range(n)) * 3,
    'wm_load': ['2'] * n + ['4'] * n + ['6'] * n,
    'rt': np.concatenate([
        complete_rt['dual_task_go_wm2_mean_rt'].values,
        complete_rt['dual_task_go_wm4_mean_rt'].values,
        complete_rt['dual_task_go_wm6_mean_rt'].values
    ])
})

aov_rt = pg.rm_anova(data=df_rt, dv='rt', within='wm_load', subject='subject', detailed=True, correction=True)
spher_rt = pg.sphericity(data=df_rt, dv='rt', within='wm_load', subject='subject')
p_rt = aov_rt['p-GG-corr'].values[0] if spher_rt.pval < 0.05 else aov_rt['p-unc'].values[0]
np2_rt = aov_rt['SS'].values[0] / (aov_rt['SS'].values[0] + aov_rt['SS'].values[1])

mse_rt = aov_rt['MS'].values[1]
results.append({
    'Analysis': 'Go Trial RT × WM Load',
    'Test': 'RM-ANOVA',
    'F': f"{aov_rt['F'].values[0]:.2f}",
    'df': f"({int(aov_rt['DF'].values[0])}, {int(aov_rt['DF'].values[1])})",
    'MSE': f"{mse_rt:.2f}",
    'p': f"{p_rt:.4f}",
    'Effect Size': f"η²p = {np2_rt:.3f}",
    'Sphericity (W, p)': f"W={spher_rt.W:.3f}, p={spher_rt.pval:.3f}",
    'n': n
})

# ============================================================================
# 3. SSRT - WM LOADS ONLY (2, 4, 6)
# ============================================================================
complete_ssrt = metrics_data_wm.dropna(subset=['SSRT_wm2', 'SSRT_wm4', 'SSRT_wm6'])
n = len(complete_ssrt)

df_ssrt_wm = pd.DataFrame({
    'subject': list(range(n)) * 3,
    'wm_load': ['2'] * n + ['4'] * n + ['6'] * n,
    'ssrt': np.concatenate([
        complete_ssrt['SSRT_wm2'].values,
        complete_ssrt['SSRT_wm4'].values,
        complete_ssrt['SSRT_wm6'].values
    ])
})

aov_ssrt_wm = pg.rm_anova(data=df_ssrt_wm, dv='ssrt', within='wm_load', subject='subject', detailed=True, correction=True)
spher_ssrt_wm = pg.sphericity(data=df_ssrt_wm, dv='ssrt', within='wm_load', subject='subject')
p_ssrt_wm = aov_ssrt_wm['p-GG-corr'].values[0] if spher_ssrt_wm.pval < 0.05 else aov_ssrt_wm['p-unc'].values[0]
np2_ssrt_wm = aov_ssrt_wm['SS'].values[0] / (aov_ssrt_wm['SS'].values[0] + aov_ssrt_wm['SS'].values[1])

mse_ssrt_wm = aov_ssrt_wm['MS'].values[1]
results.append({
    'Analysis': 'SSRT × WM Load (Dual Task Only)',
    'Test': 'RM-ANOVA',
    'F': f"{aov_ssrt_wm['F'].values[0]:.2f}",
    'df': f"({int(aov_ssrt_wm['DF'].values[0])}, {int(aov_ssrt_wm['DF'].values[1])})",
    'MSE': f"{mse_ssrt_wm:.2f}",
    'p': f"{p_ssrt_wm:.4f}",
    'Effect Size': f"η²p = {np2_ssrt_wm:.3f}",
    'Sphericity (W, p)': f"W={spher_ssrt_wm.W:.3f}, p={spher_ssrt_wm.pval:.3f}",
    'n': n
})

# ============================================================================
# 4. SSRT - ALL CONDITIONS (Simple + WM Loads)
# ============================================================================
participants_both = list(set(metrics_data_stop['prolific_id']) & set(metrics_data_wm['prolific_id']))
stop_complete = metrics_data_stop[metrics_data_stop['prolific_id'].isin(participants_both)].dropna(subset=['SSRT'])
wm_complete = metrics_data_wm[metrics_data_wm['prolific_id'].isin(participants_both)].dropna(subset=['SSRT_wm2', 'SSRT_wm4', 'SSRT_wm6'])
common = list(set(stop_complete['prolific_id']) & set(wm_complete['prolific_id']))

stop_aligned = stop_complete[stop_complete['prolific_id'].isin(common)].set_index('prolific_id')
wm_aligned = wm_complete[wm_complete['prolific_id'].isin(common)].set_index('prolific_id')
wm_aligned = wm_aligned.loc[stop_aligned.index]
n = len(common)

df_ssrt_all = pd.DataFrame({
    'subject': list(range(n)) * 4,
    'condition': ['Simple'] * n + ['WM2'] * n + ['WM4'] * n + ['WM6'] * n,
    'ssrt': np.concatenate([
        stop_aligned['SSRT'].values,
        wm_aligned['SSRT_wm2'].values,
        wm_aligned['SSRT_wm4'].values,
        wm_aligned['SSRT_wm6'].values
    ])
})

aov_ssrt_all = pg.rm_anova(data=df_ssrt_all, dv='ssrt', within='condition', subject='subject', detailed=True, correction=True)
spher_ssrt_all = pg.sphericity(data=df_ssrt_all, dv='ssrt', within='condition', subject='subject')
p_ssrt_all = aov_ssrt_all['p-GG-corr'].values[0] if spher_ssrt_all.pval < 0.05 else aov_ssrt_all['p-unc'].values[0]
np2_ssrt_all = aov_ssrt_all['SS'].values[0] / (aov_ssrt_all['SS'].values[0] + aov_ssrt_all['SS'].values[1])

mse_ssrt_all = aov_ssrt_all['MS'].values[1]
results.append({
    'Analysis': 'SSRT × Condition (Simple + WM)',
    'Test': 'RM-ANOVA',
    'F': f"{aov_ssrt_all['F'].values[0]:.2f}",
    'df': f"({int(aov_ssrt_all['DF'].values[0])}, {int(aov_ssrt_all['DF'].values[1])})",
    'MSE': f"{mse_ssrt_all:.2f}",
    'p': f"{p_ssrt_all:.4f}",
    'Effect Size': f"η²p = {np2_ssrt_all:.3f}",
    'Sphericity (W, p)': f"W={spher_ssrt_all.W:.3f}, p={spher_ssrt_all.pval:.3f}",
    'n': n
})

# ============================================================================
# 5. PROBE ACCURACY - RM ANOVA
# ============================================================================
complete_probe = metrics_data_wm.dropna(subset=[
    'probe_wm2_response_accuracy', 'probe_wm4_response_accuracy', 'probe_wm6_response_accuracy'
])
n = len(complete_probe)

df_probe = pd.DataFrame({
    'subject': list(range(n)) * 3,
    'wm_load': ['2'] * n + ['4'] * n + ['6'] * n,
    'accuracy': np.concatenate([
        complete_probe['probe_wm2_response_accuracy'].values,
        complete_probe['probe_wm4_response_accuracy'].values,
        complete_probe['probe_wm6_response_accuracy'].values
    ])
})

aov_probe = pg.rm_anova(data=df_probe, dv='accuracy', within='wm_load', subject='subject', detailed=True, correction=True)
spher_probe = pg.sphericity(data=df_probe, dv='accuracy', within='wm_load', subject='subject')
p_probe = aov_probe['p-GG-corr'].values[0] if spher_probe.pval < 0.05 else aov_probe['p-unc'].values[0]
np2_probe = aov_probe['SS'].values[0] / (aov_probe['SS'].values[0] + aov_probe['SS'].values[1])

mse_probe = aov_probe['MS'].values[1]
results.append({
    'Analysis': 'Probe Accuracy × WM Load',
    'Test': 'RM-ANOVA',
    'F': f"{aov_probe['F'].values[0]:.2f}",
    'df': f"({int(aov_probe['DF'].values[0])}, {int(aov_probe['DF'].values[1])})",
    'MSE': f"{mse_probe:.4f}",
    'p': f"{p_probe:.4f}",
    'Effect Size': f"η²p = {np2_probe:.3f}",
    'Sphericity (W, p)': f"W={spher_probe.W:.3f}, p={spher_probe.pval:.3f}",
    'n': n
})

# ============================================================================
# 6. PROBE ACCURACY BY STOP OUTCOME - Paired t-tests
# ============================================================================
successful = metrics_data_wm['probe_accuracy_on_successful_stop'].dropna()
failed = metrics_data_wm['probe_accuracy_on_failed_stop'].dropna()
go = metrics_data_wm['probe_accuracy_on_go_trials'].dropna()

# Successful vs Failed
common_sf = successful.index.intersection(failed.index)
if len(common_sf) > 2:
    t_sf, p_sf = ttest_rel(successful.loc[common_sf], failed.loc[common_sf])
    diff_sf = successful.loc[common_sf] - failed.loc[common_sf]
    d_sf = diff_sf.mean() / diff_sf.std() if diff_sf.std() > 0 else np.nan
    results.append({
        'Analysis': 'Probe Acc: Successful vs Failed Stop',
        'Test': 'Paired t-test',
        'F': f"t = {t_sf:.2f}",
        'df': f"({len(common_sf)-1})",
        'MSE': 'N/A',
        'p': f"{p_sf:.4f}",
        'Effect Size': f"d = {d_sf:.3f}",
        'Sphericity (W, p)': 'N/A',
        'n': len(common_sf)
    })

# Successful vs Go
common_sg = successful.index.intersection(go.index)
if len(common_sg) > 2:
    t_sg, p_sg = ttest_rel(successful.loc[common_sg], go.loc[common_sg])
    diff_sg = successful.loc[common_sg] - go.loc[common_sg]
    d_sg = diff_sg.mean() / diff_sg.std() if diff_sg.std() > 0 else np.nan
    results.append({
        'Analysis': 'Probe Acc: Successful Stop vs Go',
        'Test': 'Paired t-test',
        'F': f"t = {t_sg:.2f}",
        'df': f"({len(common_sg)-1})",
        'MSE': 'N/A',
        'p': f"{p_sg:.4f}",
        'Effect Size': f"d = {d_sg:.3f}",
        'Sphericity (W, p)': 'N/A',
        'n': len(common_sg)
    })

# Failed vs Go
common_fg = failed.index.intersection(go.index)
if len(common_fg) > 2:
    t_fg, p_fg = ttest_rel(failed.loc[common_fg], go.loc[common_fg])
    diff_fg = failed.loc[common_fg] - go.loc[common_fg]
    d_fg = diff_fg.mean() / diff_fg.std() if diff_fg.std() > 0 else np.nan
    results.append({
        'Analysis': 'Probe Acc: Failed Stop vs Go',
        'Test': 'Paired t-test',
        'F': f"t = {t_fg:.2f}",
        'df': f"({len(common_fg)-1})",
        'MSE': 'N/A',
        'p': f"{p_fg:.4f}",
        'Effect Size': f"d = {d_fg:.3f}",
        'Sphericity (W, p)': 'N/A',
        'n': len(common_fg)
    })

# ============================================================================
# 7. BIC MODEL COMPARISONS (using stop_wm.bic_bayes module)
# ============================================================================

# Prepare aligned data for BIC
participants_both = list(set(metrics_data_stop['prolific_id']) & set(metrics_data_wm['prolific_id']))

# Stop Success Rate BIC
stop_sr = metrics_data_stop[metrics_data_stop['prolific_id'].isin(participants_both)].dropna(subset=['stop_inhibition_success_rate'])
wm_sr = metrics_data_wm[metrics_data_wm['prolific_id'].isin(participants_both)].dropna(subset=[
    'dual_task_stop_wm2_inhibition_success_rate', 'dual_task_stop_wm4_inhibition_success_rate', 'dual_task_stop_wm6_inhibition_success_rate'])
common_sr = list(set(stop_sr['prolific_id']) & set(wm_sr['prolific_id']))
stop_sr = stop_sr[stop_sr['prolific_id'].isin(common_sr)].set_index('prolific_id')
wm_sr = wm_sr[wm_sr['prolific_id'].isin(common_sr)].set_index('prolific_id').loc[stop_sr.index]

sr_data = np.column_stack([stop_sr['stop_inhibition_success_rate'].values, 
    wm_sr['dual_task_stop_wm2_inhibition_success_rate'].values,
    wm_sr['dual_task_stop_wm4_inhibition_success_rate'].values,
    wm_sr['dual_task_stop_wm6_inhibition_success_rate'].values])
n_obs_sr = sr_data.size
bic_null_sr = calculate_bic(sr_data.flatten() - np.mean(sr_data), 1, n_obs_sr)
bic_full_sr = calculate_bic((sr_data - np.mean(sr_data, axis=0)).flatten(), 4, n_obs_sr)
delta_bic_sr = bic_null_sr - bic_full_sr

results.append({
    'Analysis': 'BIC: Stop Success Rate',
    'Test': 'Model Comparison',
    'F': f"ΔBIC = {delta_bic_sr:.2f}",
    'df': 'N/A',
    'MSE': 'N/A',
    'p': 'N/A',
    'Effect Size': interpret_bic_delta(delta_bic_sr, verbose=False),
    'Sphericity (W, p)': 'N/A',
    'n': len(common_sr)
})

# GO RT BIC
stop_rt = metrics_data_stop[metrics_data_stop['prolific_id'].isin(participants_both)].dropna(subset=['go_mean_rt'])
wm_rt = metrics_data_wm[metrics_data_wm['prolific_id'].isin(participants_both)].dropna(subset=[
    'dual_task_go_wm2_mean_rt', 'dual_task_go_wm4_mean_rt', 'dual_task_go_wm6_mean_rt'])
common_rt = list(set(stop_rt['prolific_id']) & set(wm_rt['prolific_id']))
stop_rt = stop_rt[stop_rt['prolific_id'].isin(common_rt)].set_index('prolific_id')
wm_rt = wm_rt[wm_rt['prolific_id'].isin(common_rt)].set_index('prolific_id').loc[stop_rt.index]

rt_data = np.column_stack([stop_rt['go_mean_rt'].values,
    wm_rt['dual_task_go_wm2_mean_rt'].values,
    wm_rt['dual_task_go_wm4_mean_rt'].values,
    wm_rt['dual_task_go_wm6_mean_rt'].values])
n_obs_rt_bic = rt_data.size
bic_null_rt_bic = calculate_bic(rt_data.flatten() - np.mean(rt_data), 1, n_obs_rt_bic)
bic_full_rt_bic = calculate_bic((rt_data - np.mean(rt_data, axis=0)).flatten(), 4, n_obs_rt_bic)
delta_bic_rt_bic = bic_null_rt_bic - bic_full_rt_bic

results.append({
    'Analysis': 'BIC: GO Reaction Time',
    'Test': 'Model Comparison',
    'F': f"ΔBIC = {delta_bic_rt_bic:.2f}",
    'df': 'N/A',
    'MSE': 'N/A',
    'p': 'N/A',
    'Effect Size': interpret_bic_delta(delta_bic_rt_bic, verbose=False),
    'Sphericity (W, p)': 'N/A',
    'n': len(common_rt)
})

# SSRT BIC
stop_ssrt = metrics_data_stop[metrics_data_stop['prolific_id'].isin(participants_both)].dropna(subset=['SSRT'])
wm_ssrt = metrics_data_wm[metrics_data_wm['prolific_id'].isin(participants_both)].dropna(subset=['SSRT_wm2', 'SSRT_wm4', 'SSRT_wm6'])
common_ssrt = list(set(stop_ssrt['prolific_id']) & set(wm_ssrt['prolific_id']))
stop_ssrt = stop_ssrt[stop_ssrt['prolific_id'].isin(common_ssrt)].set_index('prolific_id')
wm_ssrt = wm_ssrt[wm_ssrt['prolific_id'].isin(common_ssrt)].set_index('prolific_id').loc[stop_ssrt.index]

ssrt_data = np.column_stack([stop_ssrt['SSRT'].values,
    wm_ssrt['SSRT_wm2'].values,
    wm_ssrt['SSRT_wm4'].values,
    wm_ssrt['SSRT_wm6'].values])
n_obs_ssrt_bic = ssrt_data.size
bic_null_ssrt = calculate_bic(ssrt_data.flatten() - np.mean(ssrt_data), 1, n_obs_ssrt_bic)
bic_full_ssrt = calculate_bic((ssrt_data - np.mean(ssrt_data, axis=0)).flatten(), 4, n_obs_ssrt_bic)
delta_bic_ssrt = bic_null_ssrt - bic_full_ssrt

results.append({
    'Analysis': 'BIC: SSRT',
    'Test': 'Model Comparison',
    'F': f"ΔBIC = {delta_bic_ssrt:.2f}",
    'df': 'N/A',
    'MSE': 'N/A',
    'p': 'N/A',
    'Effect Size': interpret_bic_delta(delta_bic_ssrt, verbose=False),
    'Sphericity (W, p)': 'N/A',
    'n': len(common_ssrt)
})

# ============================================================================
# CREATE SUMMARY DATAFRAME
# ============================================================================
summary_df = pd.DataFrame(results)

print("\n")
print(summary_df.to_string(index=False))

# ============================================================================
# DESCRIPTIVE STATISTICS TABLE
# ============================================================================
print("\n\n" + "=" * 90)
print("DESCRIPTIVE STATISTICS")
print("=" * 90)

desc_results = []

# Go Trial Accuracy
for load in [2, 4, 6]:
    col = f'dual_task_go_wm{load}_choice_accuracy'
    data = metrics_data_wm[col].dropna()
    desc_results.append({
        'Measure': f'Go Accuracy (WM{load})',
        'M': f"{data.mean():.3f}",
        'SD': f"{data.std():.3f}",
        'n': len(data)
    })

# Go Trial RT
for load in [2, 4, 6]:
    col = f'dual_task_go_wm{load}_mean_rt'
    data = metrics_data_wm[col].dropna()
    desc_results.append({
        'Measure': f'Go RT (WM{load})',
        'M': f"{data.mean():.1f}ms",
        'SD': f"{data.std():.1f}ms",
        'n': len(data)
    })

# SSRT
ssrt_simple = metrics_data_stop['SSRT'].dropna()
desc_results.append({
    'Measure': 'SSRT (Simple)',
    'M': f"{ssrt_simple.mean():.1f}ms",
    'SD': f"{ssrt_simple.std():.1f}ms",
    'n': len(ssrt_simple)
})

for load in [2, 4, 6]:
    col = f'SSRT_wm{load}'
    data = metrics_data_wm[col].dropna()
    desc_results.append({
        'Measure': f'SSRT (WM{load})',
        'M': f"{data.mean():.1f}ms",
        'SD': f"{data.std():.1f}ms",
        'n': len(data)
    })

# Probe Accuracy
for load in [2, 4, 6]:
    col = f'probe_wm{load}_response_accuracy'
    data = metrics_data_wm[col].dropna()
    desc_results.append({
        'Measure': f'Probe Accuracy (WM{load})',
        'M': f"{data.mean():.3f}",
        'SD': f"{data.std():.3f}",
        'n': len(data)
    })

# Probe by Stop Outcome
desc_results.append({
    'Measure': 'Probe Acc (Successful Stop)',
    'M': f"{successful.mean():.3f}",
    'SD': f"{successful.std():.3f}",
    'n': len(successful)
})
desc_results.append({
    'Measure': 'Probe Acc (Failed Stop)',
    'M': f"{failed.mean():.3f}",
    'SD': f"{failed.std():.3f}",
    'n': len(failed)
})
desc_results.append({
    'Measure': 'Probe Acc (Go Trials)',
    'M': f"{go.mean():.3f}",
    'SD': f"{go.std():.3f}",
    'n': len(go)
})

desc_df = pd.DataFrame(desc_results)
print("\n")
print(desc_df.to_string(index=False))

# ============================================================================
# APA-FORMATTED RESULTS
# ============================================================================
print("\n\n" + "=" * 90)
print("APA-FORMATTED RESULTS (copy-paste ready)")
print("=" * 90)

print("\n--- Go Trial Performance ---")
print(f"Go trial accuracy did not differ significantly across WM load conditions, "
      f"F({int(aov_acc['DF'].values[0])}, {int(aov_acc['DF'].values[1])}) = {aov_acc['F'].values[0]:.2f}, "
      f"p = {p_acc:.3f}, η²p = {np2_acc:.3f}." if p_acc > 0.05 else 
      f"Go trial accuracy differed significantly across WM load conditions, "
      f"F({int(aov_acc['DF'].values[0])}, {int(aov_acc['DF'].values[1])}) = {aov_acc['F'].values[0]:.2f}, "
      f"p = {p_acc:.3f}, η²p = {np2_acc:.3f}.")

print(f"\nGo trial RT {'did not differ' if p_rt > 0.05 else 'differed'} significantly across WM load conditions, "
      f"F({int(aov_rt['DF'].values[0])}, {int(aov_rt['DF'].values[1])}) = {aov_rt['F'].values[0]:.2f}, "
      f"p = {p_rt:.3f}, η²p = {np2_rt:.3f}.")

print("\n--- SSRT ---")
print(f"SSRT {'did not differ' if p_ssrt_wm > 0.05 else 'differed'} significantly across WM load conditions "
      f"(within dual-task), F({int(aov_ssrt_wm['DF'].values[0])}, {int(aov_ssrt_wm['DF'].values[1])}) = {aov_ssrt_wm['F'].values[0]:.2f}, "
      f"p = {p_ssrt_wm:.3f}, η²p = {np2_ssrt_wm:.3f}.")

print(f"\nWhen including the simple stop-signal task, SSRT {'did not differ' if p_ssrt_all > 0.05 else 'differed'} "
      f"significantly across conditions, F({int(aov_ssrt_all['DF'].values[0])}, {int(aov_ssrt_all['DF'].values[1])}) = {aov_ssrt_all['F'].values[0]:.2f}, "
      f"p = {p_ssrt_all:.3f}, η²p = {np2_ssrt_all:.3f}.")

print("\n--- Probe Accuracy ---")
print(f"Probe accuracy {'did not differ' if p_probe > 0.05 else 'differed'} significantly across WM load conditions, "
      f"F({int(aov_probe['DF'].values[0])}, {int(aov_probe['DF'].values[1])}) = {aov_probe['F'].values[0]:.2f}, "
      f"p = {p_probe:.3f}, η²p = {np2_probe:.3f}.")

print(f"\nProbe accuracy did not differ between successful stops (M = {successful.mean():.3f}) and failed stops "
      f"(M = {failed.mean():.3f}), t({len(common_sf)-1}) = {t_sf:.2f}, p = {p_sf:.3f}, d = {d_sf:.3f}.")

print("\n--- BIC Model Comparisons ---")
print(f"BIC model comparisons evaluated evidence for condition effects (Kass & Raftery, 1995).")
print(f"\nStop signal success rate: ΔBIC = {delta_bic_sr:.2f} ({interpret_bic_delta(delta_bic_sr, verbose=False)} condition effects)")
print(f"Go reaction time: ΔBIC = {delta_bic_rt_bic:.2f} ({interpret_bic_delta(delta_bic_rt_bic, verbose=False)} condition effects)")
print(f"SSRT: ΔBIC = {delta_bic_ssrt:.2f} ({interpret_bic_delta(delta_bic_ssrt, verbose=False)} condition effects)")
print("\nNote: Positive ΔBIC indicates evidence FOR condition effects (full model preferred).")
print("Interpretation: 0-2 Weak, 2-6 Positive, 6-10 Strong, >10 Very Strong")

print("\n" + "=" * 90)

# ============================================================================
# EXPORT TABLES FOR GOOGLE DOCS
# ============================================================================
print("\nEXPORTING TABLES...")

# Export statistical summary to CSV
summary_df.to_csv('../data/results/statistical_summary_table.csv', index=False)
print(f"  ✓ Statistical summary saved to: data/results/statistical_summary_table.csv")

# Export descriptive statistics to CSV
desc_df.to_csv('../data/results/descriptive_statistics_table.csv', index=False)
print(f"  ✓ Descriptive statistics saved to: data/results/descriptive_statistics_table.csv")



STATISTICAL SUMMARY TABLE FOR MANUSCRIPT


                            Analysis             Test             F       df     MSE      p         Effect Size   Sphericity (W, p)  n
         Go Trial Accuracy × WM Load         RM-ANOVA          0.46  (2, 98)  0.0007 0.6340         η²p = 0.009      W=inf, p=1.000 50
               Go Trial RT × WM Load         RM-ANOVA         16.61  (2, 98)  642.40 0.0000         η²p = 0.253    W=0.901, p=0.082 50
     SSRT × WM Load (Dual Task Only)         RM-ANOVA          1.17  (2, 98) 1277.82 0.3153         η²p = 0.023    W=0.994, p=0.870 50
      SSRT × Condition (Simple + WM)         RM-ANOVA          0.37 (3, 147) 2964.71 0.6622         η²p = 0.007    W=0.342, p=0.000 50
            Probe Accuracy × WM Load         RM-ANOVA         79.24  (2, 98)  0.0017 0.0000         η²p = 0.618 W=1536.561, p=1.000 50
Probe Acc: Successful vs Failed Stop    Paired t-test      t = 1.57     (49)     N/A 0.1231           d = 0.222                 N/A 50
    Probe Ac

/Users/lyndefolsom/research/working_memory_inhibition/experiment_3/.venv/lib/python3.12/site-packages/pingouin/distribution.py:1004: RuntimeWarning: divide by zero encountered in scalar divide
  W = np.prod(eig) / (eig.sum() / d) ** d
/Users/lyndefolsom/research/working_memory_inhibition/experiment_3/.venv/lib/python3.12/site-packages/pingouin/distribution.py:1004: RuntimeWarning: divide by zero encountered in scalar divide
  W = np.prod(eig) / (eig.sum() / d) ** d
